# 📊 Fase 4 — Modelado · PANEL 1 (EDA + Clustering)
### Trabajo Final · Minería de Datos · UNMSM-FISI · 2026-I

**Pregunta:** ¿qué **perfiles de siniestralidad** existen entre los sectores y regiones del Perú?

---
### Checklist del requisito (imagen del profe)
| Contenido mínimo | ✔ |
|---|---|
| Estadísticas descriptivas | ✅ |
| Histogramas | ✅ (Plotly) |
| Mapa de correlación | ✅ |
| Boxplots | ✅ |
| **Outliers (1.5·IQR)** | ✅ |
| Agrupamiento **K-means (método del codo)** | ✅ |
| DBSCAN | ✅ (comparación) |
| **Silueta** y visualización de clusters | ✅ |
| **OBLIGATORIO:** ≥1 métrica de calidad | ✅ **silueta + inercia** |

**Librerías:** Pandas + **Plotly** (interactivo) + sklearn KMeans/DBSCAN


In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Paleta UNMSM
GRANATE, DORADO, AZUL, VERDE, GRIS = "#7a1128", "#d4a72c", "#3b6ea5", "#2e7d5b", "#6b6b6b"
PALETA = [GRANATE, DORADO, AZUL, VERDE, GRIS, "#b5651d"]
TEMPLATE = "plotly_white"

df = pd.read_csv("../data set/limpio.csv")
df = df[(df["ANIOS"] >= 2018) & (df["ANIOS"] <= 2022)]   # mismo periodo estable del Panel 2
print(f"Datos: {len(df):,} accidentes (2018–2022)")


Datos: 118,805 accidentes (2018–2022)


---
## 1️⃣ Construcción de la tabla analítica

**¿Qué vamos a agrupar?** No los accidentes individuales (serían 118 mil puntos sin sentido),
sino las **unidades región × sector económico** — cada una con su *perfil de siniestralidad*.

> *Ejemplo: "Construcción en Arequipa" es una unidad; "Minería en Pasco" es otra.*
> El clustering responde: **¿qué unidades se parecen entre sí en su forma de accidentarse?**

Filtro: solo unidades con **≥ 50 accidentes** (perfiles con soporte estadístico).


In [2]:
g = df.groupby(["REGION", "ACTIVIDAD_ECONOMICA"])

perfil = pd.DataFrame({
    "n_accidentes":     g.size(),
    "tasa_permanente":  g["PERMANENTE"].mean() * 100,
    "prop_masculino":   g["SEXO"].apply(lambda s: (s == "MASCULINO").mean() * 100),
    "prop_operario":    g["CATEGORIA_OCUPACIONAL"].apply(lambda s: s.isin(["OPERARIO","OBRERO","PEON"]).mean() * 100),
    "diversidad_formas": g["FORMA_DEL_ACCIDENTE_G"].apply(lambda s: s.nunique()),
    "concentracion_forma": g["FORMA_DEL_ACCIDENTE_G"].apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100),
}).reset_index()

perfil = perfil[perfil["n_accidentes"] >= 50].reset_index(drop=True)
perfil["log_accidentes"] = np.log10(perfil["n_accidentes"])
perfil["unidad"] = perfil["ACTIVIDAD_ECONOMICA"].str[:22] + " · " + perfil["REGION"]

print(f"Unidades región×sector a clusterizar: {len(perfil)}")
perfil.head()


Unidades región×sector a clusterizar: 85


,REGION,ACTIVIDAD_ECONOMICA,n_accidentes,tasa_permanente,prop_masculino,prop_operario,diversidad_formas,concentracion_forma,log_accidentes,unidad
0,ANCASH,"ACTIVIDADES INMOBILIARIAS, EMPRESARIALES Y DE ...",112,49.107143,92.857143,16.071429,10,39.285714,2.049218,ACTIVIDADES INMOBILIAR · ANCASH
1,ANCASH,"COMERCIO AL POR MAYOR Y AL POR MENOR, REP. VEH...",62,38.709677,96.774194,32.258065,10,20.967742,1.792392,COMERCIO AL POR MAYOR · ANCASH
2,ANCASH,CONSTRUCCIÎ,116,25.000000,93.965517,52.586207,10,23.275862,2.064458,CONSTRUCCIÎ · ANCASH
3,ANCASH,EXPLOTACIÎ DE MINAS Y CANTERAS,269,33.828996,98.884758,25.650558,10,26.765799,2.429752,EXPLOTACIÎ DE MINAS Y · ANCASH
4,ANCASH,INDUSTRIAS MANUFACTURERAS,108,28.703704,96.296296,40.740741,8,48.148148,2.033424,INDUSTRIAS MANUFACTURE · ANCASH


---
## 2️⃣ Estadísticas descriptivas *(requisito)*

In [3]:
FEATURES = ["log_accidentes","tasa_permanente","prop_masculino",
            "prop_operario","diversidad_formas","concentracion_forma"]

desc = perfil[FEATURES].describe().T
desc["mediana"] = perfil[FEATURES].median()
desc["IQR"]     = perfil[FEATURES].quantile(.75) - perfil[FEATURES].quantile(.25)
desc["skew"]    = perfil[FEATURES].skew()
display(desc[["mean","mediana","std","IQR","min","max","skew"]].round(2))


,mean,mediana,std,IQR,min,max,skew
log_accidentes,2.48,2.24,0.68,0.88,1.70,4.31,1.03
tasa_permanente,15.84,8.21,18.03,21.09,0.00,74.79,1.31
prop_masculino,83.26,91.22,20.11,12.97,16.67,100.00,-1.92
prop_operario,27.13,21.93,19.71,22.49,0.00,86.67,1.20
diversidad_formas,9.53,10.00,0.78,1.00,7.00,10.00,-1.56
concentracion_forma,37.58,33.80,14.52,13.96,20.18,83.19,1.08


## 3️⃣ Histogramas *(requisito · Plotly interactivo)*

In [4]:
fig = make_subplots(rows=2, cols=3, subplot_titles=FEATURES)
for i, c in enumerate(FEATURES):
    fig.add_trace(go.Histogram(x=perfil[c], nbinsx=20, marker_color=PALETA[i % len(PALETA)],
                               showlegend=False), row=i//3 + 1, col=i % 3 + 1)
fig.update_layout(height=520, template=TEMPLATE, title_text="Distribución de las variables del perfil",
                  title_font_color=GRANATE, bargap=0.05)
fig.show()


## 4️⃣ Boxplots y detección de outliers (**1.5·IQR**) *(requisito)*

In [5]:
fig = go.Figure()
for i, c in enumerate(FEATURES):
    fig.add_trace(go.Box(y=perfil[c], name=c, marker_color=PALETA[i % len(PALETA)],
                         boxpoints="outliers"))
fig.update_layout(height=430, template=TEMPLATE, showlegend=False,
                  title="Boxplots · outliers según la regla 1.5·IQR", title_font_color=GRANATE)
fig.show()


In [6]:
filas = []
for c in FEATURES:
    q1, q3 = perfil[c].quantile(.25), perfil[c].quantile(.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5*iqr, q3 + 1.5*iqr
    out = perfil[(perfil[c] < li) | (perfil[c] > ls)]
    filas.append({"variable": c, "Q1": q1, "Q3": q3, "IQR": iqr,
                  "límite_inf": li, "límite_sup": ls, "n_outliers": len(out)})
tabla_iqr = pd.DataFrame(filas)
display(tabla_iqr.round(2).style.hide(axis="index"))

# ¿Quiénes son los outliers de tasa_permanente? (los perfiles más peligrosos)
q1, q3 = perfil["tasa_permanente"].quantile([.25,.75])
ls = q3 + 1.5*(q3-q1)
extremos = perfil[perfil["tasa_permanente"] > ls][["unidad","n_accidentes","tasa_permanente"]]
print(f"\nUnidades ATÍPICAS por tasa de permanentes (> {ls:.1f}%):")
display(extremos.sort_values("tasa_permanente", ascending=False).round(1))
print("→ Decisión: NO se eliminan. Son perfiles REALES de alto riesgo — justo lo que buscamos detectar.")


variable,Q1,Q3,IQR,límite_inf,límite_sup,n_outliers
log_accidentes,1.950000,2.830000,0.880000,0.640000,4.150000,2
tasa_permanente,2.040000,23.120000,21.090000,-29.600000,54.760000,6
prop_masculino,82.620000,95.590000,12.970000,63.170000,115.040000,10
prop_operario,13.380000,35.870000,22.490000,-20.360000,69.600000,4
diversidad_formas,9.000000,10.000000,1.000000,7.500000,11.500000,2
concentracion_forma,26.950000,40.910000,13.960000,6.020000,61.840000,9



Unidades ATÍPICAS por tasa de permanentes (> 54.8%):


,unidad,n_accidentes,tasa_permanente
26,ADMINISTRACIÎ PLICA Y · CALLAO,119,74.8
60,PESCA · LIMA,339,59.6
32,OTRAS ACTIV. SERV. COM · CALLAO,966,57.9
33,PESCA · CALLAO,93,55.9
30,INDUSTRIAS MANUFACTURE · CALLAO,4695,55.8
35,"TRANSPORTE, ALMACENAMI · CALLAO",2694,55.3


→ Decisión: NO se eliminan. Son perfiles REALES de alto riesgo — justo lo que buscamos detectar.


## 5️⃣ Mapa de correlación *(requisito)*

In [7]:
corr = perfil[FEATURES].corr()
fig = px.imshow(corr, text_auto=".2f", aspect="auto", zmin=-1, zmax=1,
                color_continuous_scale="RdBu_r",
                title="Mapa de correlación entre variables del perfil")
fig.update_layout(height=460, template=TEMPLATE, title_font_color=GRANATE)
fig.show()

alta = [(a, b, corr.loc[a,b]) for i,a in enumerate(FEATURES) for b in FEATURES[i+1:]
        if abs(corr.loc[a,b]) > 0.7]
print("Correlaciones fuertes (|r| > 0.7):", alta if alta else "ninguna ✓ (sin redundancia)")


Correlaciones fuertes (|r| > 0.7): ninguna ✓ (sin redundancia)


---
## 6️⃣ Clustering

### 6.1 · Estandarización (Z-Score)
> **Regla del profe:** Min-Max se rompe con outliers (y aquí SÍ los hay, como vimos arriba).
> K-means usa distancias euclídeas → **es obligatorio escalar**, o `n_accidentes` dominaría todo.


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

print("Librerías de clustering cargadas ✓")


Librerías de clustering cargadas ✓


### 6.2 · Selección de variables para el clustering

No todas las variables aportan a la **separación** de grupos: algunas solo añaden ruido y
hunden la silueta. Evaluamos subconjuntos y nos quedamos con el que maximiza la calidad del
agrupamiento (esto es *selección de modelo* en aprendizaje **no supervisado**).

> Las 6 variables se conservan para el **EDA descriptivo** (arriba); el **clustering** usa
> el subconjunto que produce grupos realmente separados.


In [9]:
import itertools

candidatos = []
for r in (3, 4, 5, 6):
    for combo in itertools.combinations(FEATURES, r):
        Xc = StandardScaler().fit_transform(perfil[list(combo)])
        for k in range(2, 7):
            s = silhouette_score(Xc, KMeans(k, random_state=42, n_init=10).fit_predict(Xc))
            candidatos.append({"silueta": s, "k": k, "n_vars": r, "variables": ", ".join(combo)})

top5 = pd.DataFrame(candidatos).nlargest(5, "silueta").reset_index(drop=True)
display(top5.round(3).style.hide(axis="index"))

mejor = top5.iloc[0]
FEATURES_CLUSTER = [v.strip() for v in mejor["variables"].split(",")]
print(f"\n✅ Variables elegidas: {FEATURES_CLUSTER}")
print(f"   → silueta = {mejor['silueta']:.3f} con k = {int(mejor['k'])}")


silueta,k,n_vars,variables
0.541000,3,3,"tasa_permanente, prop_masculino, concentracion_forma"
0.536000,2,3,"prop_masculino, prop_operario, diversidad_formas"
0.525000,2,3,"tasa_permanente, prop_masculino, concentracion_forma"
0.523000,2,3,"prop_masculino, diversidad_formas, concentracion_forma"
0.512000,2,3,"tasa_permanente, prop_masculino, diversidad_formas"



✅ Variables elegidas: ['tasa_permanente', 'prop_masculino', 'concentracion_forma']
   → silueta = 0.541 con k = 3


In [10]:
X = StandardScaler().fit_transform(perfil[FEATURES_CLUSTER])
print("Datos estandarizados para el clustering:", X.shape, "| media≈0, std≈1 ✓")


Datos estandarizados para el clustering: (85, 3) | media≈0, std≈1 ✓


### 6.3 · Método del codo + Silueta *(requisito obligatorio)*
Dos métricas de calidad, no una:
- **Inercia** (codo): suma de distancias al centroide. Baja siempre → buscamos el "codo".
- **Silueta**: de −1 a 1. Mide si cada punto está bien asignado. **Buscamos el máximo.**


In [11]:
ks = range(2, 11)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X, km.labels_))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Método del CODO (inercia)",
                                                    "Coeficiente de SILUETA"))
fig.add_trace(go.Scatter(x=list(ks), y=inercias, mode="lines+markers",
                         line=dict(color=GRANATE, width=3), marker=dict(size=9)), row=1, col=1)
fig.add_trace(go.Scatter(x=list(ks), y=siluetas, mode="lines+markers",
                         line=dict(color=DORADO, width=3), marker=dict(size=9)), row=1, col=2)

K_OPT = list(ks)[int(np.argmax(siluetas))]
fig.add_vline(x=K_OPT, line_dash="dash", line_color=VERDE, row=1, col=2)
fig.add_vline(x=K_OPT, line_dash="dash", line_color=VERDE, row=1, col=1)
fig.update_xaxes(title_text="k (nº de clusters)")
fig.update_layout(height=380, template=TEMPLATE, showlegend=False,
                  title=f"Selección de k → óptimo: k = {K_OPT}", title_font_color=GRANATE)
fig.show()

print(f"k óptimo por silueta: {K_OPT}  (silueta = {max(siluetas):.3f})")
display(pd.DataFrame({"k": list(ks), "inercia": np.round(inercias,1),
                      "silueta": np.round(siluetas,3)}).set_index("k").T)


k óptimo por silueta: 3  (silueta = 0.541)


k,2,3,4,5,6,7,8,9,10
inercia,151.200,85.000,63.10,51.500,43.400,36.300,30.200,25.800,21.700
silueta,0.525,0.541,0.41,0.408,0.396,0.388,0.393,0.402,0.418


### 6.4 · K-means con el k óptimo

In [12]:
km = KMeans(n_clusters=K_OPT, random_state=42, n_init=10).fit(X)
perfil["cluster"] = km.labels_

sil = silhouette_score(X, km.labels_)
print(f"✅ MÉTRICAS DE CALIDAD (requisito obligatorio)")
print(f"   Coeficiente de SILUETA : {sil:.3f}")
print(f"   INERCIA                : {km.inertia_:.1f}")
print(f"\nTamaño de cada cluster:")
print(perfil['cluster'].value_counts().sort_index().to_string())


✅ MÉTRICAS DE CALIDAD (requisito obligatorio)
   Coeficiente de SILUETA : 0.541
   INERCIA                : 85.0

Tamaño de cada cluster:
cluster
0    60
1    17
2     8


### 6.5 · Gráfico de silueta por punto
Permite ver si algún cluster tiene puntos mal asignados (silueta negativa).


In [13]:
sv = silhouette_samples(X, km.labels_)
fig = go.Figure(); y0 = 0
for c in range(K_OPT):
    vals = np.sort(sv[km.labels_ == c])
    fig.add_trace(go.Bar(x=vals, y=np.arange(y0, y0+len(vals)), orientation="h",
                         name=f"Cluster {c}", marker_color=PALETA[c % len(PALETA)]))
    y0 += len(vals) + 4
fig.add_vline(x=sil, line_dash="dash", line_color=GRIS,
              annotation_text=f"media = {sil:.3f}")
fig.update_layout(height=520, template=TEMPLATE, bargap=0,
                  title="Silueta por unidad (cuanto más a la derecha, mejor asignada)",
                  title_font_color=GRANATE, xaxis_title="coeficiente de silueta",
                  yaxis=dict(showticklabels=False))
fig.show()


### 6.6 · Visualización de clusters con PCA *(requisito)*
Reducimos las 6 dimensiones a 2 para poder verlos.


In [14]:
pca = PCA(n_components=2, random_state=42)
comp = pca.fit_transform(X)
perfil["PC1"], perfil["PC2"] = comp[:,0], comp[:,1]
var = pca.explained_variance_ratio_

fig = px.scatter(perfil, x="PC1", y="PC2", color=perfil["cluster"].astype(str),
                 size="n_accidentes", hover_name="unidad",
                 hover_data={"tasa_permanente":":.1f", "n_accidentes":":,",
                             "PC1":False, "PC2":False, "cluster":False},
                 color_discrete_sequence=PALETA,
                 labels={"color":"Cluster"},
                 title=f"Clusters en el espacio PCA · varianza explicada: {var.sum()*100:.0f}%")
fig.update_layout(height=560, template=TEMPLATE, title_font_color=GRANATE,
                  xaxis_title=f"PC1 ({var[0]*100:.0f}%)", yaxis_title=f"PC2 ({var[1]*100:.0f}%)")
fig.show()
print("Pasa el mouse por cada punto para ver qué sector/región es (gráfico interactivo).")


Pasa el mouse por cada punto para ver qué sector/región es (gráfico interactivo).


---
## 7️⃣ Interpretación: ¿qué significa cada cluster?
El clustering solo agrupa; **darle nombre y sentido es trabajo del analista.**


In [15]:
resumen = perfil.groupby("cluster")[FEATURES_CLUSTER + ["n_accidentes"]].mean().round(1)
resumen["n_unidades"] = perfil["cluster"].value_counts().sort_index()
display(resumen.style.background_gradient(subset=["tasa_permanente"], cmap="Reds"))

print("\nEjemplos de cada cluster (los 3 de mayor volumen):")
for c in sorted(perfil["cluster"].unique()):
    sub = perfil[perfil["cluster"]==c].nlargest(3, "n_accidentes")
    print(f"\n── Cluster {c} · tasa permanente media: {resumen.loc[c,'tasa_permanente']:.1f}%")
    for _, r in sub.iterrows():
        print(f"     • {r['unidad']:45s} ({r['n_accidentes']:>5,.0f} accid., {r['tasa_permanente']:.1f}% perm.)")


,tasa_permanente,prop_masculino,concentracion_forma,n_accidentes,n_unidades
cluster,,,,,
0,8.700000,88.700000,30.600000,1614.200000,60
1,45.700000,89.100000,56.400000,801.400000,17
2,5.600000,30.300000,49.800000,778.100000,8



Ejemplos de cada cluster (los 3 de mayor volumen):

── Cluster 0 · tasa permanente media: 8.7%
     • INDUSTRIAS MANUFACTURE · LIMA                 (20,526 accid., 16.0% perm.)
     • ACTIVIDADES INMOBILIAR · LIMA                 (16,582 accid., 20.0% perm.)
     • CONSTRUCCIÎ · LIMA                            (11,063 accid., 14.4% perm.)

── Cluster 1 · tasa permanente media: 45.7%
     • INDUSTRIAS MANUFACTURE · CALLAO               (4,695 accid., 55.8% perm.)
     • TRANSPORTE, ALMACENAMI · CALLAO               (2,694 accid., 55.3% perm.)
     • ACTIVIDADES INMOBILIAR · CALLAO               (1,939 accid., 40.3% perm.)

── Cluster 2 · tasa permanente media: 5.6%
     • SERVICIOS SOCIALES Y D · LIMA                 (5,465 accid., 12.4% perm.)
     • SERVICIOS SOCIALES Y D · AREQUIPA             (  272 accid., 1.5% perm.)
     • SERVICIOS SOCIALES Y D · CALLAO               (  119 accid., 22.7% perm.)


In [16]:
# Radar: huella de cada cluster
resumen_z = (resumen[FEATURES_CLUSTER] - resumen[FEATURES_CLUSTER].mean()) / resumen[FEATURES_CLUSTER].std()
fig = go.Figure()
for c in resumen_z.index:
    fig.add_trace(go.Scatterpolar(r=resumen_z.loc[c].values, theta=FEATURES_CLUSTER,
                                  fill="toself", name=f"Cluster {c}",
                                  line_color=PALETA[c % len(PALETA)]))
fig.update_layout(height=480, template=TEMPLATE,
                  title="Huella de cada cluster (valores relativos)", title_font_color=GRANATE,
                  polar=dict(radialaxis=dict(visible=True)))
fig.show()


## 8️⃣ DBSCAN *(comparación — el profe lo menciona como alternativa)*
DBSCAN no necesita que le digas cuántos clusters hay, y además **detecta ruido** (puntos atípicos).


In [17]:
db = DBSCAN(eps=1.6, min_samples=4).fit(X)
perfil["cluster_dbscan"] = db.labels_
n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_ruido = int((db.labels_ == -1).sum())

print(f"DBSCAN → {n_cl} clusters + {n_ruido} puntos de RUIDO (outliers)")
if n_cl >= 2:
    mask = db.labels_ != -1
    print(f"Silueta DBSCAN (sin ruido): {silhouette_score(X[mask], db.labels_[mask]):.3f}")
print(f"Silueta K-means             : {sil:.3f}")

if n_ruido:
    print("\nUnidades marcadas como RUIDO (perfiles únicos, no encajan en ningún grupo):")
    display(perfil[perfil["cluster_dbscan"]==-1][["unidad","n_accidentes","tasa_permanente"]]
            .sort_values("tasa_permanente", ascending=False).round(1).head(8))


DBSCAN → 1 clusters + 1 puntos de RUIDO (outliers)
Silueta K-means             : 0.541

Unidades marcadas como RUIDO (perfiles únicos, no encajan en ningún grupo):


,unidad,n_accidentes,tasa_permanente
26,ADMINISTRACIÎ PLICA Y · CALLAO,119,74.8


### Comparación K-means vs DBSCAN
| | K-means | DBSCAN |
|---|---|---|
| Nº de clusters | hay que elegirlo (codo/silueta) | lo descubre solo |
| Detecta ruido | ❌ (fuerza a todos a un cluster) | ✅ |
| Forma de los clusters | esféricos | cualquier forma |
| **Elegido** | ✅ **por mayor silueta y clusters interpretables** | usado para detectar perfiles atípicos |


---
## 9️⃣ Ranking de riesgo (el hallazgo accionable)

In [18]:
top = perfil.nlargest(12, "tasa_permanente")
fig = px.bar(top.sort_values("tasa_permanente"), x="tasa_permanente", y="unidad",
             orientation="h", color="cluster", color_discrete_sequence=PALETA,
             text=top.sort_values("tasa_permanente")["tasa_permanente"].round(1),
             hover_data={"n_accidentes":":,"},
             title="Top 12 · unidades sector×región con mayor tasa de secuelas PERMANENTES")
fig.add_vline(x=perfil["tasa_permanente"].mean(), line_dash="dash", line_color=GRIS,
              annotation_text=f"media nacional {perfil['tasa_permanente'].mean():.1f}%")
fig.update_layout(height=520, template=TEMPLATE, title_font_color=GRANATE,
                  xaxis_title="% de accidentes con secuela permanente", yaxis_title="")
fig.show()


In [19]:
# Guardar resultados para el dashboard
perfil.to_csv("../data set/clusters_panel1.csv", index=False)
print(f"✅ Guardado: ../data set/clusters_panel1.csv  ({len(perfil)} unidades con su cluster)")


✅ Guardado: ../data set/clusters_panel1.csv  (85 unidades con su cluster)


---
## 🧾 Conclusiones del Panel 1

**Métricas de calidad (requisito obligatorio ✅):**
- **Silueta** y **Inercia** reportadas; k elegido por el **método del codo + máxima silueta**.

**Decisiones metodológicas defendibles:**
1. **Unidad de análisis:** región × sector (no accidentes sueltos) — un clustering de 118 mil
   accidentes individuales no tendría interpretación de negocio.
2. **Z-Score y no Min-Max:** hay outliers reales (detectados con 1.5·IQR) y Min-Max los aplastaría.
3. **Los outliers NO se eliminan:** son perfiles legítimos de alto riesgo — precisamente lo que
   el proyecto busca identificar.
4. **K-means sobre DBSCAN** como modelo principal (mejor silueta e interpretabilidad),
   pero DBSCAN se usa para **detectar perfiles atípicos** que no encajan en ningún grupo.

**Valor de negocio:** el ranking de unidades sector×región con mayor tasa de secuelas permanentes
es directamente accionable → **dónde debe SUNAFIL concentrar la fiscalización**.

### ➡️ Siguiente: Panel 3 (pronóstico) y Panel 4 (CRUD)
